# Summary Yahoo

Health check + coverage report for Yahoo Finance data in the local DuckDB.

Yahoo writes three tables: **`dividends`**, **`splits`** (corporate actions), and **`yahoo_prices`** (daily OHLCV).

Row-level retrieval goes through `irp.data.yahoo.prices()` / `dividends()` / `splits()`. Aggregations stay in SQL via the shared `db()` connection.

In [ ]:
import pandas as pd
from IPython.display import display

from irp.data._common import db
from irp.data.yahoo import prices, dividends, splits

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## Tables

Schema, row count, and key stats for each table written by `irp.sources.yahoo.YahooSource`.

### `dividends`

Cash dividend events. One row per (Ticker, Date). Date is `YYYYMMDD` integer.

| Column | Type | Meaning |
|---|---|---|
| Ticker | VARCHAR | Uppercase symbol (e.g. `AAPL`) |
| Date | BIGINT | Ex-dividend date as YYYYMMDD integer |
| Amount | DOUBLE | Dividend amount per share (in ticker's currency) |
| SrcId | VARCHAR | Source ticker (same as Ticker for Yahoo) |
| Src | VARCHAR | Loader name; always `yahoo` |

In [ ]:
_sample_div = dividends(tickers='AAPL')
display(_sample_div.dtypes.to_frame('dtype'))
display(_sample_div.tail())

In [ ]:
_stats_div = db().execute("""
    SELECT
        COUNT(*)               AS rows,
        COUNT(DISTINCT Ticker) AS tickers,
        MIN(Date)              AS date_min,
        MAX(Date)              AS date_max
    FROM dividends
""").df().T
_stats_div.columns = ['dividends']
display(_stats_div)

### `splits`

Stock split events. One row per (Ticker, Date). Date is `YYYYMMDD` integer.

| Column | Type | Meaning |
|---|---|---|
| Ticker | VARCHAR | Uppercase symbol |
| Date | BIGINT | Split effective date as YYYYMMDD integer |
| Ratio | DOUBLE | Split ratio (e.g. `4.0` = 4-for-1 split) |
| SrcId | VARCHAR | Source ticker |
| Src | VARCHAR | Always `yahoo` |

In [ ]:
_sample_spl = splits(tickers='AAPL')
display(_sample_spl.dtypes.to_frame('dtype'))
display(_sample_spl.tail())

In [ ]:
_stats_spl = db().execute("""
    SELECT
        COUNT(*)               AS rows,
        COUNT(DISTINCT Ticker) AS tickers,
        MIN(Date)              AS date_min,
        MAX(Date)              AS date_max
    FROM splits
""").df().T
_stats_spl.columns = ['splits']
display(_stats_spl)

### `yahoo_prices`

Daily OHLCV bars. One row per (Ticker, Date). Prices are auto-adjusted (splits + dividends) by yfinance. Date is `YYYYMMDD` integer.

| Column | Type | Meaning |
|---|---|---|
| Ticker | VARCHAR | Uppercase symbol |
| Date | BIGINT | YYYYMMDD integer |
| Open / High / Low / Close | DOUBLE | Auto-adjusted OHLC |
| Volume | BIGINT | Daily volume |

In [ ]:
_sample_yp = prices(tickers='AAPL', start='2025-04-01')
display(_sample_yp.dtypes.to_frame('dtype'))
display(_sample_yp.tail())

In [ ]:
_stats_yp = db().execute("""
    SELECT
        COUNT(*)                     AS rows,
        COUNT(DISTINCT Ticker)       AS tickers,
        MIN(Date)                    AS date_min,
        MAX(Date)                    AS date_max,
        COUNT(DISTINCT Date)         AS distinct_dates
    FROM yahoo_prices
""").df().T
_stats_yp.columns = ['yahoo_prices']
display(_stats_yp)

## Cross-table coverage

Overlap between corporate actions, prices, and company metadata.

In [ ]:
display(db().execute("""
    SELECT
        COUNT(DISTINCT d.Ticker)                                          AS dividend_tickers,
        COUNT(DISTINCT s.Ticker)                                          AS split_tickers,
        COUNT(DISTINCT p.Ticker)                                          AS price_tickers,
        COUNT(DISTINCT p.Ticker) FILTER (WHERE c.Ticker IS NOT NULL)      AS prices_with_company,
        COUNT(DISTINCT p.Ticker) FILTER (WHERE c.Ticker IS NULL)          AS prices_without_company,
        COUNT(DISTINCT d.Ticker) FILTER (WHERE p.Ticker IS NULL)          AS dividends_no_prices,
        COUNT(DISTINCT s.Ticker) FILTER (WHERE p.Ticker IS NULL)          AS splits_no_prices
    FROM yahoo_prices p
    FULL OUTER JOIN dividends  d ON p.Ticker = d.Ticker
    FULL OUTER JOIN splits     s ON p.Ticker = s.Ticker
    LEFT JOIN      companies   c ON p.Ticker = c.Ticker
""").df().T.rename(columns={0: 'count'}))
print('Tickers without company metadata: non-equity instruments and equities not covered by SimFin.')

## Freshness — when was data last updated?

Latest date per table. `yahoo_prices` freshness by market (via Stooq markets mapping).

In [ ]:
display(db().execute("""
    SELECT 'dividends'    AS tbl, MAX(Date) AS latest_date, COUNT(*) AS rows, COUNT(DISTINCT Ticker) AS tickers FROM dividends
    UNION ALL
    SELECT 'splits'       AS tbl, MAX(Date) AS latest_date, COUNT(*) AS rows, COUNT(DISTINCT Ticker) AS tickers FROM splits
    UNION ALL
    SELECT 'yahoo_prices' AS tbl, MAX(Date) AS latest_date, COUNT(*) AS rows, COUNT(DISTINCT Ticker) AS tickers FROM yahoo_prices
    ORDER BY tbl
""").df())

In [ ]:
display(db().execute("""
    SELECT
        m.Market,
        MAX(p.Date)              AS latest_date,
        COUNT(DISTINCT p.Ticker) AS tickers,
        COUNT(*)                 AS rows
    FROM yahoo_prices p
    LEFT JOIN markets m ON p.Ticker = m.Ticker
    GROUP BY m.Market
    ORDER BY latest_date DESC
""").df())

## Quality

Quick checks on `yahoo_prices`. For full anomaly review use `notebooks/review_stooq_anomalies.ipynb` (Stooq vs Yahoo divergence).

In [ ]:
display(db().execute("""
    SELECT
        COUNT(*) FILTER (WHERE Open   IS NULL) AS missing_Open,
        COUNT(*) FILTER (WHERE High   IS NULL) AS missing_High,
        COUNT(*) FILTER (WHERE Low    IS NULL) AS missing_Low,
        COUNT(*) FILTER (WHERE Close  IS NULL) AS missing_Close,
        COUNT(*) FILTER (WHERE Volume IS NULL) AS missing_Volume
    FROM yahoo_prices
""").df())

In [ ]:
_neg_tickers = db().execute(
    'SELECT DISTINCT Ticker FROM yahoo_prices WHERE Close < 0 OR Open < 0 OR High < 0 OR Low < 0'
).df()['Ticker'].tolist()
if _neg_tickers:
    print(f'Tickers with negative prices: {len(_neg_tickers)}')
    print(_neg_tickers)
else:
    print('No negative-price rows.')

In [ ]:
display(db().execute("""
    SELECT Ticker, COUNT(*) AS rows
    FROM yahoo_prices
    WHERE High < Low OR Close > High OR Close < Low
    GROUP BY Ticker
    ORDER BY rows DESC
    LIMIT 20
""").df())

## Dividends — top payers

In [ ]:
display(db().execute("""
    SELECT
        Ticker,
        COUNT(*)              AS events,
        MIN(Date)             AS first_date,
        MAX(Date)             AS last_date,
        ROUND(AVG(Amount), 4) AS avg_amount
    FROM dividends
    GROUP BY Ticker
    ORDER BY events DESC
    LIMIT 20
""").df())

## Splits — most active

In [ ]:
display(db().execute("""
    SELECT
        Ticker,
        COUNT(*)  AS events,
        MIN(Date) AS first_date,
        MAX(Date) AS last_date
    FROM splits
    GROUP BY Ticker
    ORDER BY events DESC
    LIMIT 20
""").df())